# VoiceStudio on Google Colab

Run the full [VoiceStudio](https://github.com/debpalash/VoiceStudio) app — voice cloning, voice design, video dubbing, and TTS in 646 languages — on a free Colab GPU, web UI included.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (the notebook also works on CPU, but generation is much slower).

Run the cells top to bottom. Every cell is idempotent — re-running after a hiccup is always safe.

**Contents**

- **Setup & launch (cells 1-7):** GPU check, install, backend launch, open the web UI, API smoke test.
**How this notebook works (and why):**

| Piece | Approach | Why |
|---|---|---|
| Backend install | `uv pip install --system .` | Mirrors the official Docker image: installs into Colab's Python and keeps Colab's preinstalled CUDA PyTorch when it matches our pinned torch, instead of re-downloading multi-GB wheels into a venv. The pins (`deploy/torch-constraints.txt`) are passed explicitly so torch, torchaudio and torchvision cannot drift apart (#1357). |
| Web UI | Built in-notebook with `bun` (~2 min, once per session) | Official releases ship desktop installers only — there is no prebuilt standalone web bundle to download. The backend serves the built SPA itself from `frontend/dist`. |
| Opening the app | Colab's built-in kernel port proxy | No third-party tunnel binaries; the URL is private to your Google session. (A `cloudflared` public-URL alternative is documented in the launch cell.) |

Issues with this notebook are VoiceStudio issues — report them at [debpalash/VoiceStudio/issues](https://github.com/debpalash/VoiceStudio/issues).


##Setup & launch


In [ ]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,
then re-run this notebook from the top.
(Continuing anyway works, but generation will be very slow on CPU.)


In [ ]:
# ── 2. Clone + install (idempotent; first run ~5-8 min) ─────────────────────
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/debpalash/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

# Generous network timeouts: Colab -> PyPI is usually fast, but model/wheel
# CDNs occasionally stall and uv's default timeout is short.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    """Stream a command's output; stop the cell with an actionable message on failure."""
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} (exit code {rc}).\n"
            "Scroll up for the underlying error. Re-running this cell is safe.\n"
            "Still stuck? Open an issue with the output above:\n"
            "  https://github.com/debpalash/VoiceStudio/issues"
        )

# 2a. Clone the repo (reused if already present)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present at {REPO_DIR} — reusing it.")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages (ffmpeg: audio/video processing; libsndfile1: soundfile)
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2c. bun — builds the web UI (see the intro cell for why we build it here)
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

# 2d. Build the frontend (skipped once frontend/dist/index.html exists)
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("frontend/dist already built — skipping.")
else:
    # The repo is a bun workspace: install at the repo root, build in frontend/.
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install (workspace)")
    run([BUN, "run", "--cwd", "frontend", "build"], cwd=REPO_DIR, what="frontend build (vite)")
    if not os.path.exists(dist_index):
        raise SystemExit(
            "The frontend build finished but frontend/dist/index.html is missing —\n"
            "scroll up for vite errors, then re-run this cell."
        )

# 2e. Backend dependencies — same command the official Docker image uses.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
# --constraint: `uv pip install` ignores pyproject's `[tool.uv]
# constraint-dependencies` (a project-API setting), so without it torch,
# torchaudio and torchvision resolve on their bare lower bounds and Colab's
# preinstalled torchvision can be left behind a newer torch -- which fails at
# import with "operator torchvision::nms does not exist" (#1357).
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt", "."],
    cwd=REPO_DIR, what="backend install (uv pip install --system .)")

# 2f. cuDNN 8 side-install for CTranslate2 (WhisperX ASR on GPU). PyTorch 2.8+
# ships cuDNN 9; scripts/setup.py places cuDNN 8 libs where the backend
# preloads them from (<repo>/.venv/.../cudnn8_compat). We create the .venv
# directory only so the script has its expected target — no actual venv is
# used on Colab. Non-fatal: without it, transcription falls back to the
# torch-native Whisper backend automatically.
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

# 2g. Sanity check in a fresh interpreter (this kernel may hold a stale torch).
# Imports the backend's own model stack, not just torch — Colab's system
# Python mixes preinstalled and freshly-resolved wheels, and a torchaudio or
# transformers that can't load together only shows up when the model module is
# imported (#1229). Catching it here beats a 5-minute health timeout in cell 5.
run([sys.executable, "-c",
     # Versions FIRST: if the model-stack import below fails, the cell output
     # still shows what was actually installed, which is the single most
     # useful line for diagnosing a Colab environment (#1229).
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
     "transformers {transformers.__version__}, "
     "CUDA available: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "print('Install OK - backend model stack imports cleanly')"],
    cwd=REPO_DIR, what="import sanity check")



$ git clone --depth 1 https://github.com/debpalash/VoiceStudio /content/VoiceStudio

$ apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

$ curl -fsSL https://bun.sh/install | bash

$ /root/.bun/bin/bun --version

$ /root/.bun/bin/bun install --frozen-lockfile

$ /root/.bun/bin/bun run --cwd frontend build

$ uv pip install --system --no-cache --constraint deploy/torch-constraints.txt .

$ /usr/bin/python3 /content/VoiceStudio/scripts/setup.py

$ /usr/bin/python3 -c import torch, torchaudio, uvicorn, fastapi, transformers; print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, transformers {transformers.__version__}, CUDA available: {torch.cuda.is_available()}'); from omnivoice.models.omnivoice import OmniVoice; from transformers import HiggsAudioV2TokenizerModel; print('Install OK - backend model stack imports cleanly')


In [ ]:
# ── 3. (Optional) Hugging Face token ────────────────────────────────────────
# Only needed for GATED models — e.g. pyannote speaker diarization, used by
# video dubbing for multi-speaker detection. TTS, voice cloning, and voice
# design all work WITHOUT a token, so feel free to skip this cell.
#
# To use one: accept the model terms at
#   https://huggingface.co/pyannote/speaker-diarization-3.1
# then add a Colab Secret named HF_TOKEN (key icon in the left sidebar),
# toggle "Notebook access" ON, and re-run this cell.
import os

try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.")
except Exception:
    print("No HF_TOKEN Colab Secret found — skipping. (Everything except gated "
          "diarization models works without it.)")


No HF_TOKEN Colab Secret found — skipping. (Everything except gated diarization models works without it.)


In [ ]:
# ── 4. (Optional, recommended) Pre-download the default voice model ─────────
# The default TTS engine fetches k2-fsa/OmniVoice (a few GB) on its very first
# generation. Downloading it up front makes the first request fast instead of
# a several-minute stall. Safe to re-run: already-downloaded files are reused.
from huggingface_hub import snapshot_download

path = snapshot_download("k2-fsa/OmniVoice")
print("Default TTS model cached at:", path)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Default TTS model cached at: /root/.cache/huggingface/hub/models--k2-fsa--OmniVoice/snapshots/c5fdb5ccb189668d56333f77ba2629f4cd7535f4


In [ ]:
# ── 5. Launch the backend ───────────────────────────────────────────────────
# Starts uvicorn on port 3900 and waits for /health. Re-running this cell when
# the backend is already up just reports its status.
import json
import os
import subprocess
import sys
import time
import urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info:
    print(f"Backend already running — {info}")
else:
    # Clear any half-dead process from a previous run of this cell.
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    # Headless-server deployment, same as the official Docker image: relaxes
    # the desktop-only loopback origin gate so the proxied browser session can
    # reach the settings/system routes.
    env["OMNIVOICE_SERVER_MODE"] = "1"
    # Voices, projects, and generated audio live here. Ephemeral! See the
    # troubleshooting cell for persisting it to Google Drive.
    env["OMNIVOICE_DATA_DIR"] = "/content/omnivoice_data"
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app",
         "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Backend starting (PID {proc.pid}); log: {LOG_PATH}")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break  # process died — report below
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print(f"Backend is up — {info}")
        if "cuda" not in str(info.get("device", "")):
            print("NOTE: device is not CUDA — generation will be slow. "
                  "See cell 1 to enable the GPU runtime.")
    else:
        try:
            with open(LOG_PATH, "r", errors="replace") as f:
                tail = "".join(f.readlines()[-40:])
        except OSError:
            tail = "(no log file found)"
        raise SystemExit(
            "Backend did not become healthy within 5 minutes.\n"
            f"--- last lines of {LOG_PATH} ---\n{tail}\n"
            "--- end of log ---\n"
            "Fix the error above (usually a missing dependency: re-run cell 2),\n"
            "then re-run this cell. Still stuck? Attach the log to an issue:\n"
            "  https://github.com/debpalash/VoiceStudio/issues"
        )


Backend starting (PID 4762); log: /content/omnivoice_backend.log
.....
Backend is up — {'status': 'ok', 'device': 'cpu', 'version': '0.5.0'}
NOTE: device is not CUDA — generation will be slow. See cell 1 to enable the GPU runtime.


In [ ]:
# ── 6. Open the app ─────────────────────────────────────────────────────────
# Serves localhost:3900 to YOUR browser through Colab's built-in kernel port
# proxy — authenticated to your Google session, no third-party tunnel binary.
# A new browser tab opens with the full VoiceStudio UI (allow pop-ups
# for colab.research.google.com if nothing appears).
#
# Want a PUBLIC URL instead (e.g. to open the app on your phone)? Cloudflare's
# quick tunnel works well — run in a new cell:
#   !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
#   !chmod +x /usr/local/bin/cloudflared
#   !nohup cloudflared tunnel --url http://127.0.0.1:3900 --no-autoupdate > /content/cloudflared.log 2>&1 &
#   !sleep 5 && grep -o "https://.*trycloudflare.com" /content/cloudflared.log | head -1
# Anyone with that URL can reach your session — set a share PIN in the app's
# Settings first.
!pkill -9 cloudflared 2>/dev/null || true
!sleep 1

!rm -f /content/cloudflared

!wget -q "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64" \
  -O /content/cloudflared

!chmod +x /content/cloudflared

!/content/cloudflared --version



cloudflared version 2026.8.1 (built 2026-08-13-13:51 UTC)


In [ ]:
!nohup /content/cloudflared tunnel \
  --url http://127.0.0.1:3900 \
  --no-autoupdate \
  > /content/cloudflared.log 2>&1 &
!sleep 20
!cat /content/cloudflared.log

2026-08-14T03:02:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-14T03:02:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-14T03:02:17Z INF +--------------------------------------------------------------------------------------------+
2026-08-14T03:02:17Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-14T03:02:17Z INF |  https://coupon-configuring-mixed-agreed.trycloudflare